In [ ]:
import os
os.makedirs('images', exist_ok=True)

packages = ['networkx', 'pandas', 'numpy', 'matplotlib', 'scipy']
missing_packages = []

for package in packages:
    try:
        __import__(package)
    except ImportError:
        missing_packages.append(package)

if missing_packages:
    print(f"A máquina não possui os seguintes pacotes: {', '.join(missing_packages)}")
    print("Instalando as dependências ausentes")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
    print("Instalação concluída")
else:
    print("Todas as bibliotecas necessárias já estão instaladas")

# TP: Analisando Grafos do SNAP - Epinions
**Autor:** Guilherme Antonio Merces Silva (Matrícula: 223118823)

Neste notebook, realizamos o tratamento de dados e as análises estruturais obrigatórias, bem como as execuções dos algoritmos de grafos e análises avançadas na rede Epinions.

In [ ]:
# Clean Code: Consolidação de Imports e Funções Auxiliares
import sys
from collections import Counter
import networkx as nx
import pandas as pd
import numpy as np
import scipy.sparse as sp
import scipy.stats as st
from scipy.sparse.csgraph import shortest_path
import matplotlib.pyplot as plt
import random
import time
import math
import copy

def medir_tempo(func, *args, iteracoes=30):
    tempos = []
    for _ in range(iteracoes):
        inicio = time.time()
        func(*args)
        tempos.append(time.time() - inicio)
    media = np.mean(tempos)
    dp = np.std(tempos, ddof=1) if iteracoes > 1 else 0
    if iteracoes > 1 and dp > 0:
        intervalo = st.t.interval(0.95, len(tempos)-1, loc=media, scale=dp/math.sqrt(len(tempos)))
    else:
        intervalo = (media, media)
    return media, dp, intervalo

def estimate_L(grafo, nodes_amostra):
    distancias = []
    for u in nodes_amostra:
        lengths_dict = nx.single_source_shortest_path_length(grafo, u)
        dists = [d for n, d in lengths_dict.items() if d > 0]
        if dists:
            distancias.extend(dists)
    if not distancias: return 0
    return np.mean(distancias)


## 1. Tratamento de Dados
Carregando o grafo a partir do arquivo extraído `soc-sign-epinions.txt`, convertendo-o para não-direcionado, removendo auto-loops e extraindo a Maior Componente Conexa (MCC).

In [ ]:
file_path = "soc-sign-epinions.txt"
print("[INFO] Carregando o grafo direcionado...")
G_dir = nx.read_edgelist(file_path, create_using=nx.DiGraph(), nodetype=int, data=[('sign', int)], comments='#')
print(f"Grafo Original -> Vértices: {G_dir.number_of_nodes()}, Arestas: {G_dir.number_of_edges()}")

print("[INFO] Removendo auto-loops e convertendo para não-direcionado...")
G_dir.remove_edges_from(nx.selfloop_edges(G_dir))
G = G_dir.to_undirected()

print("[INFO] Analisando Componentes Conexas...")
componentes = list(nx.connected_components(G))
tamanhos = [len(c) for c in componentes]
tamanhos.sort(reverse=True)
print(f"Número total de componentes conexas no grafo original: {len(componentes)}")
print(f"Tamanho das 10 maiores componentes: {tamanhos[:10]}")
print(f"Número de componentes unitárias (1 nó isolado): {tamanhos.count(1)}")

print("[INFO] Extraindo a Maior Componente Conexa (MCC)...")
maior_componente = max(componentes, key=len)
G_mcc = G.subgraph(maior_componente).copy()
print(f"MCC -> Vértices: {G_mcc.number_of_nodes()}, Arestas: {G_mcc.number_of_edges()}")


In [ ]:
print("--- Visualização Reduzida: Ego-Grafo do Maior Hub ---")
# Encontrar o nó com maior grau (hub)
nos_ordenados = sorted(G_mcc.degree(), key=lambda item: item[1], reverse=True)
maior_hub = nos_ordenados[0][0]
grau_hub = nos_ordenados[0][1]

print(f"Maior Hub: Nó {maior_hub} com {grau_hub} conexões.")
print("Extraindo e desenhando o Ego-Grafo (vizinhança direta)...")

ego = nx.ego_graph(G_mcc, maior_hub, radius=1)

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(ego, seed=42)

# Desenhar vizinhos em azul e o Hub em vermelho
vizinhos = [n for n in ego.nodes() if n != maior_hub]
nx.draw_networkx_nodes(ego, pos, nodelist=vizinhos, node_size=10, node_color='skyblue', alpha=0.6)
nx.draw_networkx_nodes(ego, pos, nodelist=[maior_hub], node_size=200, node_color='red')
nx.draw_networkx_edges(ego, pos, alpha=0.1, edge_color='gray')

plt.title(f"Ego-Grafo do Hub Central (Nó {maior_hub})")
plt.axis('off')
plt.savefig('images/ego_graph.png', dpi=300, bbox_inches='tight')
plt.show()


## 2. Parte I - Análise Estrutural Obrigatória
Extração de métricas topológicas fundamentais do grafo.

In [ ]:
print("[INFO] Calculando métricas básicas...")
print(f"Vértices (V): {G_mcc.number_of_nodes()}")
print(f"Arestas (E): {G_mcc.number_of_edges()}")

graus = [d for n, d in G_mcc.degree()]
print(f"\nGrau mínimo: {np.min(graus)}")
print(f"Grau máximo: {np.max(graus)}")
print(f"Grau médio: {np.mean(graus):.4f}")
print(f"Densidade: {nx.density(G_mcc):.6f}")

print("\n[WARN] Diâmetro, Triângulos e Clusterização Global suprimidos (intratabilidade computacional).")

plt.figure(figsize=(8, 5))
plt.hist(graus, bins=100, color='skyblue', edgecolor='black')
plt.title("Distribuição de Graus (Escala Logarítmica)")
plt.xlabel("Grau (k)")
plt.ylabel("Frequência")
plt.yscale('log')
plt.show()

In [ ]:

print("--- CÁLCULO DO RAIO ---")
print("Devido à natureza esparsa do grafo (E próximo a V), usaremos a matriz esparsa do SciPy.")
print("Calculando o Raio Estimado a partir da excentricidade dos 100 maiores hubs...")
start = time.time()

# 1. Obter os 100 maiores hubs da rede
degrees = dict(G_mcc.degree())
top_100_hubs = sorted(degrees, key=degrees.get, reverse=True)[:100]

# 2. Converter nós para índices da matriz
nodelist = list(G_mcc.nodes())
node_to_idx = {node: i for i, node in enumerate(nodelist)}
hub_indices = [node_to_idx[node] for node in top_100_hubs]

# 3. Criar matriz esparsa usando NetworkX
A = nx.to_scipy_sparse_array(G_mcc, nodelist=nodelist)

# 4. APSP Parcial (Busca em Largura otimizada para os Hubs)
dist_matrix = shortest_path(csgraph=A, method='D', directed=False, unweighted=True, indices=hub_indices)

# 5. A excentricidade é a maior distância de cada nó para o resto da rede
eccentricities = dist_matrix.max(axis=1)

# O Raio é a excentricidade mínima (a distância a partir do centro absoluto da rede)
raio_estimado = int(eccentricities.min())

print(f"Raio (Excentricidade do Centro): {raio_estimado}")
print(f"Tempo de execução (SciPy otimizado): {time.time() - start:.2f} segundos")


## 3. Parte II - Algoritmos da Disciplina
Medindo o tempo real de execução para algoritmos de buscas e caminhos (amostragem com 30 repetições para Intervalo de Confiança).

In [ ]:

print("[INFO] Executando algoritmos da disciplina (30 iterações para IC 95%)...")

root = list(G_mcc.nodes())[0]

# 1. BFS
bfs_media, bfs_dp, bfs_ic = medir_tempo(nx.bfs_tree, G_mcc, root, iteracoes=30)
print(f"BFS -> Média: {bfs_media:.4f}s | DP: {bfs_dp:.4f}s | IC: [{bfs_ic[0]:.4f}s, {bfs_ic[1]:.4f}s]")

# 2. DFS
dfs_media, dfs_dp, dfs_ic = medir_tempo(nx.dfs_tree, G_mcc, root, iteracoes=30)
print(f"DFS -> Média: {dfs_media:.4f}s | DP: {dfs_dp:.4f}s | IC: [{dfs_ic[0]:.4f}s, {dfs_ic[1]:.4f}s]")

# 3. Dijkstra
dijk_media, dijk_dp, dijk_ic = medir_tempo(nx.single_source_dijkstra_path_length, G_mcc, root, iteracoes=30)
print(f"Dijkstra -> Média: {dijk_media:.4f}s | DP: {dijk_dp:.4f}s | IC: [{dijk_ic[0]:.4f}s, {dijk_ic[1]:.4f}s]")

# 4. Eulerianidade
euler_media, euler_dp, euler_ic = medir_tempo(nx.is_eulerian, G_mcc, iteracoes=30)
print(f"Eulerianidade -> Média: {euler_media:.4f}s | DP: {euler_dp:.4f}s | IC: [{euler_ic[0]:.4f}s, {euler_ic[1]:.4f}s]")

# 5. Tarjan (Grafos Direcionados)
def call_tarjan():
    return list(nx.strongly_connected_components(G_dir))
tar_media, tar_dp, tar_ic = medir_tempo(call_tarjan, iteracoes=30)
print(f"Tarjan (G_dir) -> Média: {tar_media:.4f}s | DP: {tar_dp:.4f}s | IC: [{tar_ic[0]:.4f}s, {tar_ic[1]:.4f}s]")

# 6. Árvore Geradora Mínima (Kruskal)
def call_mst():
    return nx.minimum_spanning_tree(G_mcc, algorithm='kruskal')
mst_media, mst_dp, mst_ic = medir_tempo(call_mst, iteracoes=30)
print(f"MST (Kruskal) -> Média: {mst_media:.4f}s | DP: {mst_dp:.4f}s | IC: [{mst_ic[0]:.4f}s, {mst_ic[1]:.4f}s]")

print("[INFO] Algoritmos suprimidos:")
print(" -> Bellman-Ford: Inaplicável (ausência de pesos negativos).")
print(" -> Floyd-Warshall: Intratável (O(V^3) para V=119.130).")


## 4. Parte III - Propriedades Avançadas
Testando Lei de Potência, Robustez e Small-World.

In [ ]:

# 1. Lei de Potência (Power Law)
graus_contagem = Counter(graus)
x = list(graus_contagem.keys())
y = list(graus_contagem.values())

plt.figure(figsize=(8, 5))
plt.scatter(x, y, alpha=0.5, color='red')
plt.title("Lei de Potência: log(P(k)) vs log(k)")
plt.xlabel("log(Grau)")
plt.ylabel("log(Frequência)")
plt.xscale('log')
plt.yscale('log')
plt.show()



In [ ]:
print("[INFO] Teste Small-World: Gerando modelo Erdos-Renyi de referência...")
n_nodes = G_mcc.number_of_nodes()
n_edges = G_mcc.number_of_edges()
p = (2 * n_edges) / (n_nodes * (n_nodes - 1))
G_rand = nx.fast_gnp_random_graph(n_nodes, p, seed=42)

print("[INFO] Estimando métricas C e L por amostragem (N=100)...")
amostra_L = random.sample(list(G_mcc.nodes()), 100)
amostra_L_rand = random.sample(list(G_rand.nodes()), 100)

L_mcc = estimate_L(G_mcc, amostra_L)
L_rand = estimate_L(G_rand, amostra_L_rand)

amostra_C = random.sample(list(G_mcc.nodes()), 100)
amostra_C_rand = random.sample(list(G_rand.nodes()), 100)

C_mcc = np.mean([nx.clustering(G_mcc, u) for u in amostra_C])
C_rand = np.mean([nx.clustering(G_rand, u) for u in amostra_C_rand])

print(f"C(G_mcc): {C_mcc:.4f} | C(G_rand): {C_rand:.4f}")
print(f"L(G_mcc): {L_mcc:.4f} | L(G_rand): {L_rand:.4f}")

In [ ]:
n_remover = int(0.05 * G_mcc.number_of_nodes())
print(f"[INFO] Teste de Robustez: Simulação de remoção de {n_remover} nós (5%)...")

print("[INFO] Cenário 1: Falha aleatória (30 repetições)...")
mcc_sizes_rand = []
mcc_counts_rand = []
dists_rand = []
isolados_rand = []

for i in range(30):
    G_temp = G_mcc.copy()
    nos_aleatorios = random.sample(list(G_temp.nodes()), n_remover)
    G_temp.remove_nodes_from(nos_aleatorios)
    
    componentes = list(nx.connected_components(G_temp))
    maior_comp = max(componentes, key=len)
    
    mcc_sizes_rand.append(len(maior_comp))
    mcc_counts_rand.append(len(componentes))
    isolados_rand.append(sum(1 for c in componentes if len(c) == 1) / G_temp.number_of_nodes())
    
    amostra_C = random.sample(list(maior_comp), min(100, len(maior_comp)))
    dists_rand.append(estimate_L(G_temp.subgraph(maior_comp).copy(), amostra_C))

print("[INFO] Cenário 2: Ataque direcionado (remoção de hubs)...")
G_atk = G_mcc.copy()
nos_ordenados = sorted(G_atk.degree(), key=lambda item: item[1], reverse=True)
nos_alvos = [n for n, d in nos_ordenados[:n_remover]]
G_atk.remove_nodes_from(nos_alvos)

componentes_atk = list(nx.connected_components(G_atk))
maior_comp_atk = max(componentes_atk, key=len)

atk_A = len(maior_comp_atk)
atk_B = len(componentes_atk)
amostra_C_atk = random.sample(list(maior_comp_atk), min(100, len(maior_comp_atk)))
atk_C = estimate_L(G_atk.subgraph(maior_comp_atk).copy(), amostra_C_atk)
atk_D = sum(1 for c in componentes_atk if len(c) == 1) / G_atk.number_of_nodes()

fig, axs = plt.subplots(1, 4, figsize=(20, 5))
metricas_rand = [mcc_sizes_rand, mcc_counts_rand, dists_rand, isolados_rand]
metricas_atk = [atk_A, atk_B, atk_C, atk_D]
titulos = ['Métrica A: Tamanho da MCC', 'Métrica B: Número de Componentes', 'Métrica C: Distância Média', 'Métrica D: Fração Nós Isolados']

for i in range(4):
    axs[i].boxplot(metricas_rand[i], positions=[1], widths=0.4, patch_artist=True, boxprops=dict(facecolor="lightblue"))
    axs[i].axhline(y=metricas_atk[i], color='r', linestyle='--', label='Ataque Direcionado')
    axs[i].set_title(titulos[i])
    axs[i].set_xticks([])
    if i == 0: axs[i].legend()

plt.tight_layout()
plt.savefig('images/robustez_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()